In [0]:
job_health = spark.sql("""

SELECT

job_name,

COUNT(*) as total_runs,

SUM(
CASE
WHEN status='SUCCESS'
THEN 1
ELSE 0
END
) as successful_runs,

SUM(
CASE
WHEN status='FAILED'
THEN 1
ELSE 0
END
) as failed_runs,

ROUND(
100.0 *

SUM(
CASE
WHEN status='SUCCESS'
THEN 1
ELSE 0
END
)

/

COUNT(*)

,2

) as success_rate,

ROUND(
AVG(duration_minutes),
2
) as avg_runtime

FROM platform_monitoring.job_history

GROUP BY job_name

""")

In [0]:
from pyspark.sql import functions as F

job_health = (

    job_health

    .withColumn(

        "job_health_status",

        F.when(
            F.col("success_rate") < 70,
            "CRITICAL"
        )

        .when(
            F.col("success_rate") < 90,
            "WARNING"
        )

        .otherwise(
            "HEALTHY"
        )
    )
)

In [0]:
job_health.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable(
"platform_monitoring.job_health"
)

In [0]:
spark.table(
"platform_monitoring.job_health"
).show(
50,
False
)

+------------------+----------+---------------+-----------+------------+-----------+-----------------+
|job_name          |total_runs|successful_runs|failed_runs|success_rate|avg_runtime|job_health_status|
+------------------+----------+---------------+-----------+------------+-----------+-----------------+
|Customer_ETL      |3251      |2776           |475        |85.39       |29.43      |WARNING          |
|Inventory_Sync    |2929      |1619           |1310       |55.27       |29.44      |CRITICAL         |
|Sales_Aggregation |1437      |1205           |232        |83.86       |29.75      |WARNING          |
|Payment_Processing|379       |330            |49         |87.07       |29.23      |WARNING          |
|Orders_ETL        |2004      |1729           |275        |86.28       |29.73      |WARNING          |
+------------------+----------+---------------+-----------+------------+-----------+-----------------+

